# OpenAI Privacy Filter Türkçe PII Fine-tuning Colab

Bu notebook `openai/privacy-filter` modelini senin etiketlerinle fine-tune etmek, test etmek ve Hugging Face Hub'a pushlamak için hazırlanmıştır.

Kullanılan ana kaynaklar:

- GitHub repo: https://github.com/openai/privacy-filter
- Fine-tuning guide: https://github.com/openai/privacy-filter/blob/main/FINETUNING.md
- Hugging Face model: https://huggingface.co/openai/privacy-filter
- OPF output/eval dokümanları: https://github.com/openai/privacy-filter/blob/main/OUTPUT_SCHEMAS.md ve https://github.com/openai/privacy-filter/blob/main/EVAL_AND_OUTPUT_MODES.md
- Hugging Face upload guide: https://huggingface.co/docs/huggingface_hub/guides/upload

Önemli: `opf train` tam model fine-tuning yapar. T4 GPU'da veri ve context uzunluğuna bağlı olarak OOM görülebilir. İlk denemeyi küçük `N_CTX`, küçük `BATCH_SIZE` ve az epoch ile yap; ciddi eğitim için L4/A100 veya yüksek RAM GPU daha uygundur.


## 1. Runtime kontrolü

Colab menüsünden `Runtime > Change runtime type > GPU` seç. Aşağıdaki hücre GPU tipini ve Python/PyTorch durumunu gösterir.


In [18]:
import os, sys, json, platform, subprocess, textwrap, pathlib, shutil, re, random
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('CUDA version:', torch.version.cuda)
except Exception as exc:
    print('Torch import error:', exc)


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Torch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
CUDA version: 12.8


## 2. Kurulum

Bu bölüm OpenAI Privacy Filter repo'sunu clone eder, `opf` CLI paketini editable olarak kurar ve Hugging Face yardımcı paketlerini yükler.


In [2]:
%cd /content
!rm -rf /content/privacy-filter
!git clone https://github.com/openai/privacy-filter.git
%cd /content/privacy-filter
!python -m pip install -U pip setuptools wheel
!python -m pip install -e . huggingface_hub pandas scikit-learn
!opf --help


/content
Cloning into 'privacy-filter'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 76 (delta 1), reused 1 (delta 1), pack-reused 72 (from 1)
Receiving objects: 100% (76/76), 104.03 KiB | 1.86 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/privacy-filter
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 128.6 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency

## 3. Çalışma klasörleri ve Drive opsiyonu

`USE_DRIVE=True` yaparsan veri, checkpoint ve çıktıların Google Drive altında kalır. Aksi halde Colab runtime kapanınca dosyalar silinir.


In [19]:
USE_DRIVE = False  # Drive kullanmak için True yap
PROJECT_NAME = 'privacy_filter_tr_pii'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR = Path('/content/drive/MyDrive') / PROJECT_NAME
else:
    WORK_DIR = Path('/content') / PROJECT_NAME

DATA_DIR = WORK_DIR / 'data'
OUTPUT_DIR = WORK_DIR / 'finetuned_checkpoint'
PRED_DIR = WORK_DIR / 'predictions'
BASE_SNAPSHOT_DIR = WORK_DIR / 'base_openai_privacy_filter_snapshot'
BASE_CHECKPOINT_DIR = BASE_SNAPSHOT_DIR / 'original'

for p in [WORK_DIR, DATA_DIR, OUTPUT_DIR, PRED_DIR, BASE_CHECKPOINT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('WORK_DIR:', WORK_DIR)
print('DATA_DIR:', DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)


WORK_DIR: /content/privacy_filter_tr_pii
DATA_DIR: /content/privacy_filter_tr_pii/data
OUTPUT_DIR: /content/privacy_filter_tr_pii/finetuned_checkpoint


## 4. Base checkpoint indirme

Repo README'ine göre `opf`, checkpoint verilmezse varsayılan modeli indirip kullanabilir. Burada daha kontrollü olması için Hugging Face'teki `openai/privacy-filter` snapshot'ını lokal klasöre indiriyoruz.


In [20]:
from huggingface_hub import snapshot_download

# Dikkat: HF repo kökü Transformers formatında; OPF CLI eğitim için `original/` altındaki
# OPF-native checkpoint gerekir. Kök config.json ile `encoding` hatası alınır.
snapshot_download(
    repo_id='openai/privacy-filter',
    repo_type='model',
    local_dir=str(BASE_SNAPSHOT_DIR),
    local_dir_use_symlinks=False,
    allow_patterns=['original/*'],
)

config_path = BASE_CHECKPOINT_DIR / 'config.json'
config = json.loads(config_path.read_text(encoding='utf-8'))
if not config.get('encoding'):
    raise ValueError(f'Wrong checkpoint config: {config_path} has no encoding. Use the original/ OPF checkpoint folder.')

print('OPF base checkpoint:', BASE_CHECKPOINT_DIR)
print('encoding:', config.get('encoding'))
print('Base checkpoint files:')
for item in sorted(BASE_CHECKPOINT_DIR.iterdir()):
    print(' -', item.name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

OPF base checkpoint: /content/privacy_filter_tr_pii/base_openai_privacy_filter_snapshot/original
encoding: o200k_base
Base checkpoint files:
 - config.json
 - dtypes.json
 - model.safetensors
 - viterbi_calibration.json


## 5. `label_space.json` oluşturma

Senin etiket setin aşağıdaki gibi tanımlanıyor. OPF dokümanında `span_class_names` tercih edilen alan; `O` background sınıfı ilk sırada olmalı.


In [21]:
LABELS = [
    'O',
    'tckn',
    'secret',
    'iban',
    'vkn',
    'account_number',
    'private_address',
    'private_date',
    'private_phone',
    'private_email',
    'private_person',
]

label_space = {
    'category_version': 'tr_privacy_v1',
    'span_class_names': LABELS,
}

LABEL_SPACE_PATH = DATA_DIR / 'label_space.json'
LABEL_SPACE_PATH.write_text(json.dumps(label_space, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(LABEL_SPACE_PATH)
print(LABEL_SPACE_PATH.read_text(encoding='utf-8'))


/content/privacy_filter_tr_pii/data/label_space.json
{
  "category_version": "tr_privacy_v1",
  "span_class_names": [
    "O",
    "tckn",
    "secret",
    "iban",
    "vkn",
    "account_number",
    "private_address",
    "private_date",
    "private_phone",
    "private_email",
    "private_person"
  ]
}



## 6. Dataset formatı

OPF `train` ve `eval` JSON/JSONL dosyası bekler. Her satır bir örnek olmalı.

Desteklenen ana format:

```json
{"text":"Ahmet Yılmaz TCKN 12345678901","label":[{"category":"private_person","start":0,"end":12},{"category":"tckn","start":18,"end":29}]}
```

Alternatif `spans` formatı da desteklenir:

```json
{"text":"Ahmet Yılmaz TCKN 12345678901","spans":{"private_person: Ahmet Yılmaz":[[0,12]],"tckn: 12345678901":[[18,29]]}}
```

`start` dahil, `end` hariç karakter offset'idir. Türkçe karakterler Python string indeksine göre sayılır.


In [22]:
# Küçük sentetik örnek. Gerçek eğitim için kendi dosyanı yükleyeceksin.
# Bu hücre sadece formatın doğru olduğunu göstermek için var.
examples = [
    {
        'text': 'Ahmet Yılmaz TCKN 12345678901, telefon 05551234567, email ahmet@example.com.',
        'label': [
            {'category': 'private_person', 'start': 0, 'end': 12},
            {'category': 'tckn', 'start': 18, 'end': 29},
            {'category': 'private_phone', 'start': 39, 'end': 50},
            {'category': 'private_email', 'start': 58, 'end': 75},
        ],
    },
    {
        'text': 'VKN 1234567890 olan şirketin IBAN bilgisi TR330006100519786457841326.',
        'label': [
            {'category': 'vkn', 'start': 4, 'end': 14},
            {'category': 'iban', 'start': 42, 'end': 68},
        ],
    },
    {
        'text': 'Adres: Atatürk Mah. No: 12 Ankara. Kart/account no ACC-TR-998877.',
        'label': [
            {'category': 'private_address', 'start': 7, 'end': 33},
            {'category': 'account_number', 'start': 51, 'end': 64},
        ],
    },
]

sample_path = DATA_DIR / 'sample_format.jsonl'
with sample_path.open('w', encoding='utf-8') as f:
    for row in examples:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
print(sample_path)
print(sample_path.read_text(encoding='utf-8'))


/content/privacy_filter_tr_pii/data/sample_format.jsonl
{"text": "Ahmet Yılmaz TCKN 12345678901, telefon 05551234567, email ahmet@example.com.", "label": [{"category": "private_person", "start": 0, "end": 12}, {"category": "tckn", "start": 18, "end": 29}, {"category": "private_phone", "start": 39, "end": 50}, {"category": "private_email", "start": 58, "end": 75}]}
{"text": "VKN 1234567890 olan şirketin IBAN bilgisi TR330006100519786457841326.", "label": [{"category": "vkn", "start": 4, "end": 14}, {"category": "iban", "start": 42, "end": 68}]}
{"text": "Adres: Atatürk Mah. No: 12 Ankara. Kart/account no ACC-TR-998877.", "label": [{"category": "private_address", "start": 7, "end": 33}, {"category": "account_number", "start": 51, "end": 64}]}



## 7. Dataset yükleme

İki yol var:

1. Colab'a `train.jsonl` / `validation.jsonl` / `test.jsonl` yükle.
2. Dosyaların Drive'daysa path değişkenlerini doğrudan ayarla.

Aşağıdaki hücre uploaded dosyaları `DATA_DIR` içine kopyalar. Sadece tek dosyan varsa onu yükle; sonraki hücre otomatik train/validation/test split yapabilir.


In [10]:
from google.colab import files

UPLOAD_DATA = True  # dosya yüklemek için True yap

if UPLOAD_DATA:
    uploaded = files.upload()
    for name, content in uploaded.items():
        target = DATA_DIR / name
        target.write_bytes(content)
        print('uploaded ->', target)
else:
    print('UPLOAD_DATA=False. Dosyaların hazırsa aşağıdaki path değişkenlerini ayarla.')


Saving all_data.jsonl to all_data.jsonl
uploaded -> /content/privacy_filter_tr_pii/data/all_data.jsonl


In [23]:
# Kendi dosya pathlerini burada ayarla.
# Örnek: RAW_DATASET_PATH = Path('/content/drive/MyDrive/privacy_dataset/all.jsonl')
RAW_DATASET_PATH = DATA_DIR / 'all_data.jsonl'
TRAIN_PATH = DATA_DIR / 'train.jsonl'
VALIDATION_PATH = DATA_DIR / 'validation.jsonl'
TEST_PATH = DATA_DIR / 'test.jsonl'

print('RAW_DATASET_PATH:', RAW_DATASET_PATH)
print('exists:', RAW_DATASET_PATH.exists())


RAW_DATASET_PATH: /content/privacy_filter_tr_pii/data/all_data.jsonl
exists: True


## 8. Dataset doğrulama, normalize etme ve split

Bu hücre:

- JSONL okur.
- `label` ve `spans` formatlarını normalize eder.
- Etiketlerin `label_space.json` içinde olup olmadığını kontrol eder.
- Offset sınırlarını doğrular.
- İstersen tek dosyayı train/validation/test olarak böler.


In [24]:
ALLOWED_LABELS = set(LABELS) - {'O'}


def read_jsonl(path: Path):
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f'{path}:{line_no} invalid JSON: {exc}')
    return rows


def spans_to_label_list(spans):
    labels = []
    if not isinstance(spans, dict):
        raise ValueError('spans must be an object')
    for key, offsets in spans.items():
        category = str(key).split(': ', 1)[0]
        for start, end in offsets:
            labels.append({'category': category, 'start': int(start), 'end': int(end)})
    return labels


def normalize_record(row, idx):
    if 'text' not in row:
        raise ValueError(f'row {idx}: missing text')
    text = str(row['text'])
    if 'label' in row and row.get('label') is not None:
        labels = row['label']
    elif 'spans' in row and row.get('spans') is not None:
        labels = spans_to_label_list(row['spans'])
    else:
        labels = []
    if not isinstance(labels, list):
        raise ValueError(f'row {idx}: label must be list')

    normalized = []
    for j, item in enumerate(labels):
        if not isinstance(item, dict):
            raise ValueError(f'row {idx} label {j}: must be object')
        category = str(item.get('category', '')).strip()
        if category not in ALLOWED_LABELS:
            raise ValueError(f'row {idx} label {j}: unknown category {category!r}')
        start = item.get('start')
        end = item.get('end')
        if isinstance(start, bool) or isinstance(end, bool) or not isinstance(start, int) or not isinstance(end, int):
            raise ValueError(f'row {idx} label {j}: start/end must be integers')
        if not (0 <= start <= end <= len(text)):
            raise ValueError(f'row {idx} label {j}: invalid span ({start}, {end}) for text len {len(text)}')
        if start == end:
            continue
        normalized.append({'category': category, 'start': start, 'end': end})

    normalized.sort(key=lambda x: (x['start'], x['end'], x['category']))
    for a, b in zip(normalized, normalized[1:]):
        if b['start'] < a['end']:
            print(f'WARNING row {idx}: overlapping spans: {a} and {b}')

    return {'text': text, 'label': normalized}


def write_jsonl(path: Path, rows):
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


def split_rows(rows, train_ratio=0.8, val_ratio=0.1, seed=42):
    rows = list(rows)
    rng = random.Random(seed)
    rng.shuffle(rows)
    n = len(rows)
    n_train = max(1, int(n * train_ratio)) if n >= 3 else max(1, n - 1)
    n_val = max(1, int(n * val_ratio)) if n >= 10 else (1 if n >= 3 else 0)
    train = rows[:n_train]
    val = rows[n_train:n_train+n_val]
    test = rows[n_train+n_val:]
    if not val and len(train) > 1:
        val = [train.pop()]
    if not test and len(train) > 1:
        test = [train.pop()]
    return train, val, test

raw_rows = read_jsonl(RAW_DATASET_PATH)
rows = [normalize_record(row, i) for i, row in enumerate(raw_rows)]
print('Total rows:', len(rows))

CREATE_SPLIT_FROM_RAW = True
if CREATE_SPLIT_FROM_RAW:
    train_rows, val_rows, test_rows = split_rows(rows)
    write_jsonl(TRAIN_PATH, train_rows)
    write_jsonl(VALIDATION_PATH, val_rows)
    write_jsonl(TEST_PATH, test_rows)
else:
    # Eğer train/validation/test zaten hazırsa normalize edip tekrar yaz.
    for src, dst in [(TRAIN_PATH, TRAIN_PATH), (VALIDATION_PATH, VALIDATION_PATH), (TEST_PATH, TEST_PATH)]:
        normalized = [normalize_record(row, i) for i, row in enumerate(read_jsonl(src))]
        write_jsonl(dst, normalized)

for path in [TRAIN_PATH, VALIDATION_PATH, TEST_PATH]:
    print(path, 'rows=', len(read_jsonl(path)))


Total rows: 103923
/content/privacy_filter_tr_pii/data/train.jsonl rows= 83138
/content/privacy_filter_tr_pii/data/validation.jsonl rows= 10392
/content/privacy_filter_tr_pii/data/test.jsonl rows= 10393


## 9. Offset görsel kontrolü

Eğitimden önce birkaç örnekte spanların doğru karakterleri işaretlediğini kontrol et. Offset hatası NER eğitiminde en yaygın sorundur.


In [25]:
def preview_record(row):
    text = row['text']
    print('TEXT:', text)
    for span in row['label']:
        frag = text[span['start']:span['end']]
        print(f"  {span['category']:16s} [{span['start']:>3}, {span['end']:>3}] -> {frag!r}")

for row in read_jsonl(TRAIN_PATH)[:5]:
    preview_record(row)
    print('-' * 80)


TEXT: Cari hesap kaydinda vergi no 1421978094 tanimlandi.
  vkn              [ 29,  39] -> '1421978094'
--------------------------------------------------------------------------------
TEXT: Arama yapılacak numara listesine 0539-101-20-30 cep telefonu kaydı eklendi.
  private_phone    [ 33,  47] -> '0539-101-20-30'
--------------------------------------------------------------------------------
TEXT: Hasta kimlik alanı sorgusu sistemde 111111***10 şeklinde sonuçlandı.
  tckn             [ 36,  47] -> '111111***10'
--------------------------------------------------------------------------------
TEXT: Adayın doğum tarihi 09.09.1992 şeklinde kimlikte yer alıyor.
  private_date     [ 20,  30] -> '09.09.1992'
--------------------------------------------------------------------------------
TEXT: Vekaletname kaydında adı soyadı Halil Işık olarak kaydedildi.
  private_person   [ 32,  42] -> 'Halil Işık'
--------------------------------------------------------------------------------


## 10. Eğitim parametreleri

İlk deneme için güvenli ayarlar:

- `N_CTX=512` veya `1024`: uzun belge varsa artırılabilir ama GPU belleği artar.
- `BATCH_SIZE=1`: Colab için daha güvenli.
- `GRAD_ACCUM_STEPS`: efektif batch'i artırır.
- `EPOCHS`: küçük veri için 3-10 denenebilir; gerçek veri için validasyon kaybına bak.


In [26]:
DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
N_CTX = 512
EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

print({
    'DEVICE': DEVICE,
    'N_CTX': N_CTX,
    'EPOCHS': EPOCHS,
    'BATCH_SIZE': BATCH_SIZE,
    'GRAD_ACCUM_STEPS': GRAD_ACCUM_STEPS,
    'LEARNING_RATE': LEARNING_RATE,
})


{'DEVICE': 'cuda', 'N_CTX': 512, 'EPOCHS': 3, 'BATCH_SIZE': 1, 'GRAD_ACCUM_STEPS': 8, 'LEARNING_RATE': 1e-05}


## 11. Eğitim

Bu hücre `opf train` çalıştırır. Çıktı klasöründe `config.json`, `model.safetensors`, `finetune_summary.json` ve `USAGE.txt` oluşur.


In [27]:
import os
os.environ['OPF_TRAIN_PROGRESS_INTERVAL_S'] = '15'

!rm -rf "{OUTPUT_DIR}"
!mkdir -p "{OUTPUT_DIR}"

!opf train "{TRAIN_PATH}"   --validation-dataset "{VALIDATION_PATH}"   --checkpoint "{BASE_CHECKPOINT_DIR}"   --label-space-json "{LABEL_SPACE_PATH}"   --output-dir "{OUTPUT_DIR}"   --overwrite-output   --device "{DEVICE}"   --n-ctx {N_CTX}   --epochs {EPOCHS}   --batch-size {BATCH_SIZE}   --grad-accum-steps {GRAD_ACCUM_STEPS}   --learning-rate {LEARNING_RATE}   --weight-decay {WEIGHT_DECAY}   --max-grad-norm {MAX_GRAD_NORM}

print('Output files:')
for p in sorted(OUTPUT_DIR.iterdir()):
    print(' -', p.name, p.stat().st_size, 'bytes')


info: rebuilt output head for target label space (41 labels; copied_rows=41; exact=29; fallback=12)
training plan: epochs=3 train_examples=83138 train_windows=83138 validation_examples=10392 validation_windows=10392 progress_interval_s=15.0
train progress: epoch=1/3 batch=624/83138 windows=624/83138 examples_seen=624/83138 tokens=14477 train_loss=1.346240 train_token_accuracy=0.7940 eta_epoch=33:11 eta_total=01:40:04
train progress: epoch=1/3 batch=1440/83138 windows=1440/83138 examples_seen=1440/83138 tokens=33631 train_loss=0.857225 train_token_accuracy=0.8383 eta_epoch=28:32 eta_total=01:26:35
train progress: epoch=1/3 batch=2256/83138 windows=2256/83138 examples_seen=2256/83138 tokens=52452 train_loss=0.650039 train_token_accuracy=0.8669 eta_epoch=27:01 eta_total=01:22:35
train progress: epoch=1/3 batch=3072/83138 windows=3072/83138 examples_seen=3072/83138 tokens=71401 train_loss=0.530333 train_token_accuracy=0.8874 eta_epoch=26:11 eta_total=01:20:34
train progress: epoch=1/3 batc

## 12. Eğitim özetini inceleme

`finetune_summary.json` en iyi epoch, loss, token accuracy ve label-space bilgisini içerir.


In [28]:
summary_path = OUTPUT_DIR / 'finetune_summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps({
    'best_epoch': summary.get('best_epoch'),
    'best_metric_name': summary.get('best_metric_name'),
    'best_metric': summary.get('best_metric'),
    'num_train_examples': summary.get('num_train_examples'),
    'num_validation_examples': summary.get('num_validation_examples'),
    'span_class_names': summary.get('span_class_names'),
}, ensure_ascii=False, indent=2))

print('Epoch metrics:')
for e in summary.get('epoch_metrics', []):
    print(e)

{
  "best_epoch": 3,
  "best_metric_name": "validation_loss",
  "best_metric": 0.002157915852249276,
  "num_train_examples": 83138,
  "num_validation_examples": 10392,
  "span_class_names": [
    "O",
    "tckn",
    "secret",
    "iban",
    "vkn",
    "account_number",
    "private_address",
    "private_date",
    "private_phone",
    "private_email",
    "private_person"
  ]
}
Epoch metrics:
{'elapsed_s': 1593.6850067229998, 'epoch': 1, 'optimizer_steps': 10393, 'train_batches': 83138, 'train_loss': 0.03890803883171036, 'train_token_accuracy': 0.9907683947678255, 'train_tokens': 1932275, 'validation_batches': 10392, 'validation_loss': 0.003961100774073546, 'validation_token_accuracy': 0.9990998430919151, 'validation_tokens': 242180}
{'elapsed_s': 1587.528100984, 'epoch': 2, 'optimizer_steps': 10393, 'train_batches': 83138, 'train_loss': 0.002713801173860853, 'train_token_accuracy': 0.9994633269074019, 'train_tokens': 1932275, 'validation_batches': 10392, 'validation_loss': 0.002180

## 13. Test/eval çalıştırma

`opf eval` ground-truth test dosyasında metrik üretir. Custom label-space ile eğittiğimiz için test seti aynı label isimlerini kullanıyorsa `typed` eval uygundur.


In [29]:
METRICS_OUT = PRED_DIR / 'test_metrics.json'
PREDICTIONS_OUT = PRED_DIR / 'test_predictions.jsonl'
TIMINGS_OUT = PRED_DIR / 'test_timings.json'

!opf eval "{TEST_PATH}"   --checkpoint "{OUTPUT_DIR}"   --device "{DEVICE}"   --n-ctx {N_CTX}   --eval-mode typed   --per-class   --label-counts   --metrics-out "{METRICS_OUT}"   --predictions-out "{PREDICTIONS_OUT}"   --timings-out "{TIMINGS_OUT}"   --preview

print('metrics exists:', METRICS_OUT.exists())
if METRICS_OUT.exists():
    print(METRICS_OUT.read_text(encoding='utf-8')[:4000])


summary:
  field                        value    
  ---------------------------  ---------
  examples                     10393    
  tokens                       241020   
  windows                      10393    
  window_tokens                241020   
  padded_window_tokens         241020   
  elapsed_s                    82.0727  
  tokens_per_s                 2936.6655
  window_tokens_per_s          2936.6655
  padded_window_tokens_per_s   2936.6655
  eval_mode                    typed    
  inference_tokens_per_second  3027.1969
  loss                         0.0028   
  token_accuracy               0.9996   

detection_metrics:
  metric                    value 
  ------------------------  ------
  detection.precision       0.9998
  detection.recall          0.9996
  detection.f1              0.9997
  detection.f2              0.9996
  detection.span.precision  0.9988
  detection.span.recall     0.9978
  detection.span.f1         0.9983
  detection.span.f2         0.9980

per_c

## 14. Serbest metin testi

Bu hücre eğitilmiş checkpoint ile doğrudan metin üzerinde inference yapar. JSON output içinde `detected_spans` ve `redacted_text` alanlarını görürsün.


In [30]:
TEST_TEXT = 'Mehmet Kaya TCKN 12345678901, VKN 1234567890, IBAN TR330006100519786457841326, telefon 05551234567, e-posta mehmet@example.com.'

!opf --checkpoint "{OUTPUT_DIR}" --device "{DEVICE}" --n-ctx {N_CTX} --format json --output-mode typed "{TEST_TEXT}"


summary: output_mode=typed spans=5 by_label=iban:4, private_email:1 latency_ms=484.6 decoded_mismatch=no
{
  "schema_version": 1,
  "summary": {
    "output_mode": "typed",
    "span_count": 5,
    "by_label": {
      "iban": 4,
      "private_email": 1
    },
    "decoded_mismatch": false
  },
  "text": "Mehmet Kaya TCKN 12345678901, VKN 1234567890, IBAN TR330006100519786457841326, telefon 05551234567, e-posta mehmet@example.com.",
  "detected_spans": [
    {
      "label": "iban",
      "start": 17,
      "end": 28,
      "text": "12345678901",
      "placeholder": "<IBAN>"
    },
    {
      "label": "iban",
      "start": 34,
      "end": 44,
      "text": "1234567890",
      "placeholder": "<IBAN>"
    },
    {
      "label": "iban",
      "start": 51,
      "end": 77,
      "text": "TR330006100519786457841326",
      "placeholder": "<IBAN>"
    },
    {
      "label": "iban",
      "start": 87,
      "end": 98,
      "text": "05551234567",
      "placeholder": "<IBAN>"
    },
   

## 15. Test dokümanı yükleyip redaction yapma

Burada `.txt` dosyası yükleyebilir veya Drive path'i verebilirsin. OPF `-f` ile tüm dosyayı tek input olarak işler.


In [32]:
UPLOAD_TEST_DOCUMENT = False  # txt dosyası yüklemek için True yap
TEST_DOCUMENT_PATH = DATA_DIR / 'test_document.txt'

if UPLOAD_TEST_DOCUMENT:
    uploaded = files.upload()
    first_name = next(iter(uploaded))
    TEST_DOCUMENT_PATH = DATA_DIR / first_name
    TEST_DOCUMENT_PATH.write_bytes(uploaded[first_name])
else:
    TEST_DOCUMENT_PATH.write_text(TEST_TEXT + 'Adres: Atatürk Mah. No: 12 Ankara. Secret: sk-demo-1234567890abcdef', encoding='utf-8')

print('TEST_DOCUMENT_PATH:', TEST_DOCUMENT_PATH)
print(TEST_DOCUMENT_PATH.read_text(encoding='utf-8')[:1000])


TEST_DOCUMENT_PATH: /content/privacy_filter_tr_pii/data/test_document.txt
Mehmet Kaya TCKN 12345678901, VKN 1234567890, IBAN TR330006100519786457841326, telefon 05551234567, e-posta mehmet@example.com.Adres: Atatürk Mah. No: 12 Ankara. Secret: sk-demo-1234567890abcdef


In [41]:

  DOC_JSON_OUT = PRED_DIR / 'document_prediction.json'
  DOC_RAW_OUT = PRED_DIR / 'document_prediction_raw_stdout.txt'
  DOC_REDACTED_OUT = PRED_DIR / 'document_redacted.txt'

  import subprocess, json

  cmd = [
      'opf', '--checkpoint', str(OUTPUT_DIR), '--device', DEVICE, '--n-ctx',
  str(N_CTX),
      '--format', 'json', '--output-mode', 'typed', '-f',
  str(TEST_DOCUMENT_PATH)
  ]

  proc = subprocess.run(cmd, text=True, capture_output=True, check=True)

  # Ham stdout'u sakla
  DOC_RAW_OUT.write_text(proc.stdout, encoding='utf-8')

  print(proc.stderr)
  print(proc.stdout[:4000])

  # stdout içindeki ilk JSON objesini parse et
  stdout = proc.stdout.lstrip()
  payload, end_idx = json.JSONDecoder().raw_decode(stdout)

  # JSON'dan sonra gelen color legend kısmını yok say
  extra = stdout[end_idx:].strip()
  if extra:
      print('Non-JSON OPF output after JSON was ignored:')
      print(extra[:1000])

  DOC_JSON_OUT.write_text(
      json.dumps(payload, ensure_ascii=False, indent=2) + '\n',
      encoding='utf-8'
  )

  DOC_REDACTED_OUT.write_text(
      payload.get('redacted_text', ''),
      encoding='utf-8'
  )

  print('Raw stdout:', DOC_RAW_OUT)
  print('JSON:', DOC_JSON_OUT)
  print('Redacted text:', DOC_REDACTED_OUT)

summary: output_mode=typed spans=7 by_label=iban:4, private_address:1, private_email:1, tckn:1 latency_ms=477.7 decoded_mismatch=no

{
  "schema_version": 1,
  "summary": {
    "output_mode": "typed",
    "span_count": 7,
    "by_label": {
      "iban": 4,
      "private_address": 1,
      "private_email": 1,
      "tckn": 1
    },
    "decoded_mismatch": false
  },
  "text": "Mehmet Kaya TCKN 12345678901, VKN 1234567890, IBAN TR330006100519786457841326, telefon 05551234567, e-posta mehmet@example.com.Adres: Atatürk Mah. No: 12 Ankara. Secret: sk-demo-1234567890abcdef",
  "detected_spans": [
    {
      "label": "iban",
      "start": 17,
      "end": 28,
      "text": "12345678901",
      "placeholder": "<IBAN>"
    },
    {
      "label": "tckn",
      "start": 34,
      "end": 44,
      "text": "1234567890",
      "placeholder": "<TCKN>"
    },
    {
      "label": "iban",
      "start": 51,
      "end": 77,
      "text": "TR330006100519786457841326",
      "placeholder": "<IBAN>"
 

## 16. Hugging Face model kartı hazırlama

Bu bölüm eğitilmiş checkpoint klasörüne `README.md` ve `label_space.json` ekler. Push sonrası kullanıcılar modeli `snapshot_download` ile indirip `opf --checkpoint` ile kullanabilir.


In [42]:
shutil.copy2(LABEL_SPACE_PATH, OUTPUT_DIR / 'label_space.json')

readme = f'''---
license: apache-2.0
base_model: openai/privacy-filter
tags:
- token-classification
- pii
- privacy
- openai-privacy-filter
- turkish
---

# Turkish Privacy Filter Fine-tune

This checkpoint is a fine-tuned OpenAI Privacy Filter model for Turkish-oriented privacy labels.

Base model: `openai/privacy-filter`

Labels:

```json
{json.dumps(label_space, ensure_ascii=False, indent=2)}
```

## Local usage

```bash
git clone https://github.com/openai/privacy-filter.git
cd privacy-filter
pip install -e .
python -c "from huggingface_hub import snapshot_download; snapshot_download(repo_id='YOUR_USERNAME/YOUR_REPO', local_dir='tr_privacy_filter')"
opf --checkpoint ./tr_privacy_filter --device cuda --format json "Mehmet Kaya TCKN 12345678901"
```

## Artifacts

- `config.json`
- `model.safetensors`
- `finetune_summary.json`
- `USAGE.txt`
- `label_space.json`

## Notes

Privacy Filter is a privacy redaction aid, not a legal anonymization guarantee. Evaluate on in-domain Turkish data before production use.
'''

(OUTPUT_DIR / 'README.md').write_text(readme, encoding='utf-8')
print((OUTPUT_DIR / 'README.md').read_text(encoding='utf-8')[:3000])


---
license: apache-2.0
base_model: openai/privacy-filter
tags:
- token-classification
- pii
- privacy
- openai-privacy-filter
- turkish
---

# Turkish Privacy Filter Fine-tune

This checkpoint is a fine-tuned OpenAI Privacy Filter model for Turkish-oriented privacy labels.

Base model: `openai/privacy-filter`

Labels:

```json
{
  "category_version": "tr_privacy_v1",
  "span_class_names": [
    "O",
    "tckn",
    "secret",
    "iban",
    "vkn",
    "account_number",
    "private_address",
    "private_date",
    "private_phone",
    "private_email",
    "private_person"
  ]
}
```

## Local usage

```bash
git clone https://github.com/openai/privacy-filter.git
cd privacy-filter
pip install -e .
python -c "from huggingface_hub import snapshot_download; snapshot_download(repo_id='YOUR_USERNAME/YOUR_REPO', local_dir='tr_privacy_filter')"
opf --checkpoint ./tr_privacy_filter --device cuda --format json "Mehmet Kaya TCKN 12345678901"
```

## Artifacts

- `config.json`
- `model.safetensors

## 17. Hugging Face'e login ve push

Hugging Face token'ını Colab Secrets içine `HF_TOKEN` adıyla koyabilir veya `login()` widget'ına yapıştırabilirsin. `write` yetkisi olan token gerekir.


In [38]:
from huggingface_hub import login, HfApi

DO_HF_LOGIN = True  # push yapmadan önce True yap
if DO_HF_LOGIN:
    login()  # Colab/Jupyter içinde token widget açar
else:
    print('DO_HF_LOGIN=False. Push için True yap veya HF_TOKEN secret/env kullan.')


HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Hugging Face login tamamlandı.")
else:
    print("WARNING: HF_TOKEN bulunamadı. Public model indirme çalışabilir, Hub push kapalı kalmalı.")




Hugging Face login tamamlandı.


In [43]:
from huggingface_hub import HfApi

PUSH_TO_HUB = True  # Gerçek push için True yap
HF_REPO_ID = 'BTX24/privacy-filter-tr-pii'  # örn: boran/privacy-filter-tr-pii
HF_PRIVATE = True

if PUSH_TO_HUB:
    api = HfApi()
    api.create_repo(repo_id=HF_REPO_ID, repo_type='model')
    api.upload_folder(
        folder_path=str(OUTPUT_DIR),
        repo_id=HF_REPO_ID,
        repo_type='model',
        commit_message='Upload Turkish privacy-filter fine-tuned checkpoint',
        allow_patterns=[
            'config.json',
            'model.safetensors',
            'finetune_summary.json',
            'USAGE.txt',
            'label_space.json',
            'README.md',
        ],
    )
    print('Uploaded:', f'https://huggingface.co/{HF_REPO_ID}')
else:
    print('PUSH_TO_HUB=False. HF_REPO_ID ayarlayıp True yapınca upload_folder çalışır.')


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ckpoint/model.safetensors:   8%|7         |  216MB / 2.80GB            

Uploaded: https://huggingface.co/BTX24/privacy-filter-tr-pii


## 18. Hub'dan indirip tekrar test etme

Push yaptıktan sonra bu hücre modeli Hub'dan temiz bir klasöre indirir ve inference çalıştırır. Böylece repo'ya gerçekten kullanılabilir checkpoint yüklendiğini doğrularsın.


In [45]:
VERIFY_FROM_HUB = True
DOWNLOADED_CHECKPOINT = WORK_DIR / 'downloaded_from_hub'

if VERIFY_FROM_HUB:
    if DOWNLOADED_CHECKPOINT.exists():
        shutil.rmtree(DOWNLOADED_CHECKPOINT)
    snapshot_download(repo_id=HF_REPO_ID, repo_type='model', local_dir=str(DOWNLOADED_CHECKPOINT), local_dir_use_symlinks=False)
    cmd = [
        'opf', '--checkpoint', str(DOWNLOADED_CHECKPOINT), '--device', DEVICE, '--n-ctx', str(N_CTX),
        '--format', 'json', '--output-mode', 'typed', TEST_TEXT,
    ]
    proc = subprocess.run(cmd, text=True, capture_output=True, check=True)
    print(proc.stderr)
    print(proc.stdout)
else:
    print('VERIFY_FROM_HUB=False. Push sonrası True yap.')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

summary: output_mode=typed spans=5 by_label=iban:4, private_email:1 latency_ms=446.1 decoded_mismatch=no

{
  "schema_version": 1,
  "summary": {
    "output_mode": "typed",
    "span_count": 5,
    "by_label": {
      "iban": 4,
      "private_email": 1
    },
    "decoded_mismatch": false
  },
  "text": "Mehmet Kaya TCKN 12345678901, VKN 1234567890, IBAN TR330006100519786457841326, telefon 05551234567, e-posta mehmet@example.com.",
  "detected_spans": [
    {
      "label": "iban",
      "start": 17,
      "end": 28,
      "text": "12345678901",
      "placeholder": "<IBAN>"
    },
    {
      "label": "iban",
      "start": 34,
      "end": 44,
      "text": "1234567890",
      "placeholder": "<IBAN>"
    },
    {
      "label": "iban",
      "start": 51,
      "end": 77,
      "text": "TR330006100519786457841326",
      "placeholder": "<IBAN>"
    },
    {
      "label": "iban",
      "start": 87,
      "end": 98,
      "text": "05551234567",
      "placeholder": "<IBAN>"
    },
  

In [49]:
import subprocess, json, textwrap

KALLAVI_TEST_TEXT = """
Hasta Mehmet Kaya, 14.03.2025 tarihinde Ankara Çankaya'daki Atatürk
Mahallesi
No: 12 Daire: 5 adresinden başvuru yaptı. TCKN bilgisi 12345678901 olarak,
VKN bilgisi ise 1234567890 olarak kaydedildi.

Ödeme için verilen IBAN: TR330006100519786457841326.
Alternatif IBAN bilgisi TR120001200945800013456789 olarak sistemde duruyor.
İletişim telefonu 0555 123 45 67, ikinci telefon +90 532 987 65 43.
E-posta adresi mehmet.kaya@example.com, yedek e-posta m.kaya@kurum.com.tr.

Hasta yakını Ayşe Demir için TCKN 10987654321 girildi.
Adres bilgisi: Cumhuriyet Cad. Sağlık Apt. No: 34 Kat: 3 İzmir Konak.
Hesap numarası ACC-TR-2025-998877 ve müşteri/account_number değeri
445566778899.

Sistem entegrasyonu için secret değerleri şunlardır:
sk-live-abc1234567890xyz, api_key_987654321_SECRET ve token-TR-demo-778899.

Raporu hazırlayan kişi Dr. Selin Arslan olup, işlem tarihi 22 Mayıs
2026'dır.
""".strip()

cmd = [
    'opf',
    '--checkpoint', str(DOWNLOADED_CHECKPOINT),
    '--device', DEVICE,
    '--n-ctx', str(N_CTX),
    '--format', 'json',
    '--output-mode', 'typed',
    KALLAVI_TEST_TEXT,
]

proc = subprocess.run(cmd, text=True, capture_output=True, check=True)

print("STDERR:")
print(proc.stderr)

stdout = proc.stdout.lstrip()

# OPF bazen JSON'dan sonra color legend basıyor; ilk JSON objesini ayıklıyoruz.
payload, end_idx = json.JSONDecoder().raw_decode(stdout)
extra = stdout[end_idx:].strip()

print("SUMMARY:")
print(json.dumps(payload.get("summary", {}), ensure_ascii=False, indent=2))

print("\nDETECTED SPANS:")
for span in payload.get("detected_spans", []):
    print(
        f"{span['label']:18s} "
        f"[{span['start']:>4}, {span['end']:>4}] "
        f"{span['text']!r} -> {span.get('placeholder')}"
    )

print("\nREDACTED TEXT:")
print(payload.get("redacted_text", ""))

if extra:
    print("\nIGNORED NON-JSON OUTPUT:")
    print(extra[:1000])

KALLAVI_JSON_OUT = PRED_DIR / 'kallavi_test_prediction.json'
KALLAVI_REDACTED_OUT = PRED_DIR / 'kallavi_test_redacted.txt'

KALLAVI_JSON_OUT.write_text(
    json.dumps(payload, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8'
)

KALLAVI_REDACTED_OUT.write_text(
    payload.get("redacted_text", ""),
    encoding='utf-8'
)

print("JSON:", KALLAVI_JSON_OUT)
print("Redacted:", KALLAVI_REDACTED_OUT)

STDERR:
summary: output_mode=typed spans=18 by_label=account_number:2, iban:5, private_address:2, private_date:2, private_email:2, private_phone:1, secret:3, tckn:1 latency_ms=480.9 decoded_mismatch=no

SUMMARY:
{
  "output_mode": "typed",
  "span_count": 18,
  "by_label": {
    "account_number": 2,
    "iban": 5,
    "private_address": 2,
    "private_date": 2,
    "private_email": 2,
    "private_phone": 1,
    "secret": 3,
    "tckn": 1
  },
  "decoded_mismatch": false
}

DETECTED SPANS:
private_date       [  19,   29] '14.03.2025' -> <PRIVATE_DATE>
iban               [ 133,  144] '12345678901' -> <IBAN>
tckn               [ 169,  179] '1234567890' -> <TCKN>
iban               [ 225,  251] 'TR330006100519786457841326' -> <IBAN>
iban               [ 277,  303] 'TR120001200945800013456789' -> <IBAN>
iban               [ 347,  361] '0555 123 45 67' -> <IBAN>
private_phone      [ 378,  395] '+90 532 987 65 43' -> <PRIVATE_PHONE>
private_email      [ 412,  435] 'mehmet.kaya@example.com' 

In [50]:
  try:
      from google.colab import runtime
      runtime.unassign()
  except Exception:
      import os
      os.kill(os.getpid(), 9)

## 19. Sorun giderme

- `CUDA out of memory`: `N_CTX=256/512`, `BATCH_SIZE=1`, daha az epoch kullan veya L4/A100 seç.
- `Unknown entity label`: dataset içindeki kategori `label_space.json` içinde yoktur.
- `invalid span`: `start/end` offsetleri metin uzunluğunun dışında veya yanlış hesaplanmıştır.
- `output-dir already contains files`: notebook zaten `--overwrite-output` kullanıyor; yine de eski klasörü silmek gerekebilir.
- `Hugging Face auth` hatası: token'ın `write` yetkisi olduğundan ve `HF_REPO_ID` değerinin `username/repo_name` formatında olduğundan emin ol.
